# Prediksi Hipertensi

Notebook ini dikhususkan untuk memprediksi **Hipertensi** menggunakan model Machine Learning (XGBoost).
Penjelasan tahapan:
1. **Load Data**: Membaca dataset `hypertension_data.csv`.
2. **Preprocessing**: Memilih fitur (gejala/metrik) yang relevan dan membersihkan data kosong.
3. **Training**: Menerapkan SMOTE jika data tidak seimbang, lalu melatih XGBoost.
4. **Evaluasi**: Mengukur akurasi, sensitivitas, dan melihat Confusion Matrix.
5. **Inference**: Cara memprediksi data pasien baru.


## 1. Imports & Setup
Mengimpor library utama yang dibutuhkan untuk manipulasi data dan pembuatan model.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import matplotlib.pyplot as plt


## 2. Load Dataset
Membaca dataset `hypertension_data.csv` ke dalam Pandas DataFrame.


In [ ]:
dataset_path = 'hypertension_data.csv'
if dataset_path.endswith('.arff'):
    data, meta = arff.loadarff(dataset_path)
    df = pd.DataFrame(data)
    # Decode byte strings to normal strings
    for col in df.select_dtypes([object]):
        df[col] = df[col].str.decode('utf-8')
else:
    df = pd.read_csv(dataset_path)

df.head()


## 3. Preprocessing
Menyesuaikan format kolom ke format standar (`core_*`) agar model dapat mengenali fitur dengan benar.


In [ ]:
# Mapping fitur hypertension_data.csv
df['core_age'] = df['age']
df['core_systolic_bp'] = df['trestbps']
df['core_cholesterol'] = df['chol']
df['core_heart_rate'] = df['thalach']

df['core_bmi'] = np.nan
df['core_glucose'] = np.nan

# Target
df['label_hypertension'] = df['target']


## 4. Model Training
Memilih fitur yang relevan, menangani missing values (median imputation), membagi data menjadi Train & Test, dan menyeimbangkan kelas dengan SMOTE sebelum melatih model XGBoost.


In [ ]:
features = ['core_age', 'core_systolic_bp', 'core_cholesterol', 'core_heart_rate', 'core_bmi', 'core_glucose']
target = 'label_hypertension'

# Ambil fitur yang tersedia di dataframe
available_features = [f for f in features if f in df.columns]

X = df[available_features].copy()
y = df[target].copy()

# Handle missing values (Isi dengan nilai median)
X = X.fillna(X.median())

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Cek rasio kelas, terapkan SMOTE jika minoritas terlalu kecil (< 40%)
minority_ratio = y_train.value_counts(normalize=True).min()
if minority_ratio < 0.4:
    print(f"Data tidak seimbang (minority: {minority_ratio:.1%}), mengaplikasikan SMOTE...")
    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)
else:
    print("Data cukup seimbang, tidak perlu SMOTE.")

# Latih XGBoost
model = xgb.XGBClassifier(
    n_estimators=100, 
    max_depth=5, 
    learning_rate=0.1, 
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train, y_train)
print("Model berhasil dilatih!")


## 5. Evaluasi Performa
Melihat seberapa baik model ini membedakan antara pasien sehat dan yang berisiko.


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC Score: {auc:.4f}")

fig, ax = plt.subplots(figsize=(5,4))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Sehat", "Risiko Tinggi"])
disp.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix')
plt.show()


## 6. Feature Importance
Fitur/metrik tubuh mana yang paling berperan besar dalam prediksi model?


In [ ]:
fi = pd.Series(model.feature_importances_, index=available_features).sort_values(ascending=True)
plt.figure(figsize=(8, 5))
fi.plot(kind='barh', color='teal')
plt.title(f"Feature Importance - {disease_name}")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()


## 7. Inference (Prediksi Data Baru)
Contoh penggunaan model untuk memprediksi probabilitas risiko pada pasien baru.


In [ ]:
# Contoh input dari pasien
user_input = {
    'core_age': 55,
    'core_systolic_bp': 140,
    'core_cholesterol': 220,
    'core_heart_rate': 85,
    'core_bmi': 28.5,
    'core_glucose': 150
}

# Siapkan input DataFrame sesuai dengan urutan fitur model
input_df = pd.DataFrame([user_input])
for col in available_features:
    if col not in input_df.columns:
        input_df[col] = np.nan # Isi dengan NaN jika tidak ada
        
# Urutkan kolom sesuai training
input_df = input_df[available_features]
# Imputasi jika perlu (untuk demo ini kita asumsikan pakai training median, di prod pakai scaler/imputer dari train)
input_df = input_df.fillna(X.median())

# Prediksi Probabilitas
prob = model.predict_proba(input_df)[0][1]
print(f"Probabilitas Pasien Terkena {disease_name}: {prob:.1%}")
